# Fine-tune Llama 3.1 8B with LoRA (Auto-Download Version)

This notebook fine-tunes Meta's Llama 3.1 8B model using Unsloth and LoRA on the Python Code Instructions dataset.

**Key Features:**
- Auto-downloads dataset if not present
- Auto-downloads model if not present
- Pre-training and post-training inference tests
- Memory-efficient LoRA training
- GPU monitoring and statistics
- Model merging and saving capabilities

## Step 0: Auto-Download Dataset & Model (NEW)

This cell automatically downloads the training dataset and model if they don't exist locally.

In [16]:
import os
from datasets import load_dataset
from huggingface_hub import snapshot_download
import shutil

print("\n" + "="*70)
print("STEP 0: AUTO-DOWNLOAD DATASET & MODEL")
print("="*70)

# ============================================================================
# AUTO-DOWNLOAD DATASET
# ============================================================================
dataset_path = "./python_code_instructions_18k_alpaca"
dataset_hf_path = os.path.join(dataset_path, "hf_format")

if not os.path.exists(dataset_hf_path):
    print("\n📥 Dataset not found. Downloading...")
    print(f"   Source: iamtarun/python_code_instructions_18k_alpaca")
    print(f"   Target: {dataset_path}")
    
    try:
        # Download from Hugging Face
        dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
        
        # Create directory and save
        os.makedirs(dataset_path, exist_ok=True)
        dataset.save_to_disk(dataset_hf_path)
        
        print(f"   ✅ Dataset downloaded successfully!")
        print(f"   Size: {len(dataset):,} examples")
    except Exception as e:
        print(f"   ❌ Error downloading dataset: {e}")
        raise
else:
    print(f"\n✅ Dataset already exists at {dataset_hf_path}")

# ============================================================================
# AUTO-DOWNLOAD MODEL
# ============================================================================
model_path = "./Meta-Llama-3.1-8B-bnb-4bit"

if not os.path.exists(model_path) or len(os.listdir(model_path)) < 3:
    print("\n📥 Model not found. Downloading...")
    print(f"   Source: unsloth/Meta-Llama-3.1-8B-bnb-4bit")
    print(f"   Target: {model_path}")
    
    try:
        snapshot_download(
            repo_id="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
            local_dir=model_path,
            resume_download=True  # Resume if interrupted
        )
        print(f"   ✅ Model downloaded successfully!")
    except Exception as e:
        print(f"   ❌ Error downloading model: {e}")
        raise
else:
    print(f"\n✅ Model already exists at {model_path}")

print("\n" + "="*70)
print("✅ ALL DOWNLOADS COMPLETE - Ready for training!")
print("="*70 + "\n")


STEP 0: AUTO-DOWNLOAD DATASET & MODEL

✅ Dataset already exists at ./python_code_instructions_18k_alpaca/hf_format

✅ Model already exists at ./Meta-Llama-3.1-8B-bnb-4bit

✅ ALL DOWNLOADS COMPLETE - Ready for training!



## Step 1: Import Libraries

In [17]:
from unsloth import FastLanguageModel
import torch
import os
from transformers import TextStreamer
from datasets import load_from_disk
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

## Step 2: Configuration

In [18]:
# Model configuration
max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True 

# Alpaca prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Test example for inference
instruction = "Create a function to calculate the sum of a sequence of integers."
test_input = "[1, 2, 3, 4, 5]"

## Step 3: Load Model & Test Before Training

In [19]:
print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "./Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("\n" + "="*70)
print("BEFORE TRAINING - Base Model Inference")
print("="*70)

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction,
        test_input,
        "",  # Leave blank for generation
    )
], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1000)

Loading model...
==((====))==  Unsloth 2026.7.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load ./Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.



BEFORE TRAINING - Base Model Inference
<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.

### Instruction:
Create a function to calculate the sum of a sequence of integers.

### Input:
[1, 2, 3, 4, 5]

### Response:


Both `max_new_tokens` (=1000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


15<|end_of_text|>


## Step 4: Load & Format Dataset

In [20]:
print("\nLoading dataset...")
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Load dataset from disk (auto-downloaded above)
# IMPORTANT: Dataset is saved to hf_format subdirectory, so load from there
dataset_path = "./python_code_instructions_18k_alpaca/hf_format"

# Check if path exists and provide helpful error
if not os.path.exists(dataset_path):
    print(f"❌ ERROR: Dataset not found at {dataset_path}")
    print(f"   Please run Step 0 (Auto-Download) first!")
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

dataset_on_disk = load_from_disk(dataset_path)
dataset = dataset_on_disk.map(formatting_prompts_func, batched=True)

print(f"✅ Dataset loaded: {len(dataset):,} examples")


Loading dataset...
✅ Dataset loaded: 18,612 examples


## Step 5: Setup LoRA

In [21]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## Step 6: Configure Trainer

In [22]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 25,  # Change to num_train_epochs for full training
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        # FIX: Disable checkpoint saving to avoid pickling errors
        save_strategy = "no",  # Don't save intermediate checkpoints
        save_steps = 0,  # Explicitly set to 0
        # Note: Use save_strategy="epoch" or save_steps=N if you want checkpoints after fixing pickling
    ),
)

## Step 7: Monitor GPU & Train

In [23]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")
print("\nStarting training...\n")

trainer_stats = trainer.train()

GPU = NVIDIA GB10. Max memory = 121.693 GB.
11.604 GB of memory reserved.

Starting training...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,612 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.595784
2,1.624706
3,1.266803
4,1.447473
5,1.402866
6,1.200181
7,0.894853
8,0.815733
9,0.991495
10,0.573514


## Step 8: Training Statistics

In [24]:
# Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print("\n" + "="*70)
print("TRAINING COMPLETE - Statistics")
print("="*70)
print(f"⏱️  Training time: {trainer_stats.metrics['train_runtime']} seconds")
print(f"⏱️  Training time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")
print(f"💾 Peak memory used: {used_memory} GB")
print(f"💾 Memory for LoRA: {used_memory_for_lora} GB")
print(f"📊 Memory usage: {used_percentage}% of max ({lora_percentage}% for LoRA)")
print("="*70)


TRAINING COMPLETE - Statistics
⏱️  Training time: 86.5335 seconds
⏱️  Training time: 1.44 minutes
💾 Peak memory used: 12.756 GB
💾 Memory for LoRA: 1.152 GB
📊 Memory usage: 10.482% of max (0.947% for LoRA)


## Step 9: Test After Training

In [25]:
print("\n" + "="*70)
print("AFTER TRAINING - Fine-tuned Model Inference")
print("="*70)

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction,
        test_input,
        "",  # Leave blank for generation
    )
], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1000)


AFTER TRAINING - Fine-tuned Model Inference
<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.

### Instruction:
Create a function to calculate the sum of a sequence of integers.

### Input:
[1, 2, 3, 4, 5]

### Response:


Both `max_new_tokens` (=1000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/prabir/dgx-book/.venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/prabir/dgx-book/.venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Futu

def sum_sequence(sequence):
    total = 0
    for number in sequence:
        total += number
    return total

sum_sequence([1, 2, 3, 4, 5])<|end_of_text|>


## Step 10: Save Model (LoRA + Merged)

In [26]:
print("\n" + "="*70)
print("SAVING MODEL")
print("="*70)

# Save LoRA adapters
print("\n1️⃣  Saving LoRA adapters...")
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("   ✅ Saved to ./lora_model")

# Save merged model
print("\n2️⃣  Saving merged model (16-bit)...")
model.save_pretrained_merged("model_merged", tokenizer, save_method="merged_16bit")
print("   ✅ Saved to ./model_merged")

print("\n" + "="*70)
print("✅ MODEL SAVED SUCCESSFULLY")
print("="*70)
print("\nYou can now:")
print("  - Use ./lora_model for inference with LoRA")
print("  - Use ./model_merged for standalone inference")
print("  - Push to HuggingFace Hub if desired")


SAVING MODEL

1️⃣  Saving LoRA adapters...
   ✅ Saved to ./lora_model

2️⃣  Saving merged model (16-bit)...


/home/prabir/dgx-book/.venv/lib/python3.12/site-packages/unsloth_zoo/saving_utils.py:2929: UserWarning: Base model should be a 16bits or mxfp4 base model for a 16bit model merge. Use `save_method=forced_merged_4bit` instead
  warnings.warn("Base model should be a 16bits or mxfp4 base model for a 16bit model merge. Use `save_method=forced_merged_4bit` instead")


   ✅ Saved to ./model_merged

✅ MODEL SAVED SUCCESSFULLY

You can now:
  - Use ./lora_model for inference with LoRA
  - Use ./model_merged for standalone inference
  - Push to HuggingFace Hub if desired
